# tbats_jax — daily panel on Colab T4

This is the *realistic* TBATS workload: **T=730 (2 years of daily data)** with weekly + yearly seasonality. On real retail/utility panels this is typical. The earlier T=1500 benchmark was half-hourly data, an unusual format that stressed the GPU's scan serialization hard.

**Setup:** Runtime → Change runtime type → T4 GPU.

**Measured CPU baseline** (Apple Silicon M-series, single core):

| N    | compile | warm  | per-series |
|------|---------|-------|------------|
|   32 |  2.4 s  | 1.6 s | 50.4 ms    |
|  100 |  5.4 s  | 4.7 s | 46.6 ms    |
|  500 | 24.9 s  | 24 s  | 48.2 ms    |
| 1000 | 44.2 s  | 43 s  | 43.2 ms    |

At ~45 ms/series on CPU, GPU has headroom to actually win.

## 1. Install

In [ ]:
!pip install -q --upgrade "jax[cuda12]" optimistix

## 2. Upload `tbats_jax_colab.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile, os
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content')
        print('extracted to /content/tbats_jax')

assert os.path.isdir('/content/tbats_jax')

In [ ]:
%cd /content/tbats_jax
!pip install -q -e .

## 3. Verify GPU

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
print('backend :', jax.default_backend())
print('devices :', jax.devices())
assert jax.default_backend() == 'gpu', 'GPU not active'

## 4. Headline run: N=1000

Compile cost ~30-60 s (T=730 graph is half the size of T=1500). Warm wall should be ~10-20 s on T4 → 10-20 ms/series vs CPU's 43 ms.

Expected speedup: **2-4×**.

In [ ]:
from benchmarks.colab_daily_panel import run
r1000 = run(N=1000, T=730)

## 5. Push: N=10000

Memory footprint at T=730, N=10000, float64: ~60 MB y, ~250 MB autodiff intermediates. Comfortably fits T4's 16 GB. CPU extrapolation: ~430 s. GPU target: **30-60 s**.

In [ ]:
r10000 = run(N=10000, T=730)

## 6. Comparison

In [ ]:
cpu_per_series_ms = 45.0  # measured at N=1000 on Apple Silicon M-series CPU
print(f'{"N":>6} | {"GPU compile":>12} | {"GPU warm":>10} | {"GPU ms/ser":>12} | {"CPU equiv":>10} | {"speedup":>9}')
print('-' * 78)
for N, r in [(1000, r1000), (10000, r10000)]:
    cpu_eq = N * cpu_per_series_ms / 1000
    speedup = cpu_eq / r['warm_wall']
    print(f'{N:>6} | {r["compile_time"]:>11.1f}s | {r["warm_wall"]:>9.2f}s | {r["per_series_ms"]:>11.2f} | {cpu_eq:>9.1f}s | {speedup:>8.1f}x')